# 04 — XAI: LIME vs SHAP on IndoBERT Sentiment Model

Comparing two explainability methods on the fine-tuned IndoBERT model:

| Method | Type | What it does |
|--------|------|-------------|
| **LIME** | Local, perturbation-based | Fits a local linear model by perturbing input tokens |
| **SHAP** | Local + global, game-theory-based | Computes Shapley values for each token's contribution |

**Goal:** Analyze which words drive Positif / Negatif / Netral predictions.

In [ ]:
## 0. Google Colab Setup  (skip if running locally)
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab detected — mounting Drive & installing packages …")

    from google.colab import drive
    drive.mount("/content/drive")

    # ── Adjust DRIVE_PROJECT to match your Drive folder path ──────────
    DRIVE_PROJECT = "/content/drive/MyDrive/xai_lime_vs_shap"

    if os.path.isdir(DRIVE_PROJECT):
        os.chdir(DRIVE_PROJECT)
        print(f"CWD → {DRIVE_PROJECT}")
    else:
        print(f"WARNING: '{DRIVE_PROJECT}' not found on Drive.")
        print("Option B — clone from GitHub instead (edit URL, then uncomment):")
        print("  !git clone https://github.com/YOUR/xai_lime_vs_shap.git /content/xai_lime_vs_shap")
        print("  import os; os.chdir('/content/xai_lime_vs_shap')")

    # Install required packages
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers", "torch", "datasets", "scikit-learn",
         "accelerate", "lime", "shap"],
        check=True,
    )
    print("Packages ready.")
else:
    print("Local environment — no Colab setup needed.")


In [ ]:
## 1. Install XAI libraries (run once)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "lime", "shap", "matplotlib", "--quiet"])
print("lime & shap ready")

In [ ]:
## 2. Imports & Load Model

import json, warnings
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import lime
from lime.lime_text import LimeTextExplainer
import shap

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")


def find_project_root() -> Path:
    """Find repository root robustly across VS Code / Jupyter / Colab."""
    markers = ["data", "notebooks", "src"]
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path("/content/xai_lime_vs_shap"),
    ]
    for candidate in candidates:
        if all((candidate / m).exists() for m in markers):
            return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()
MODEL_DIR    = PROJECT_ROOT / "outputs" / "indobert_sentiment"
DATA_PATH    = PROJECT_ROOT / "data" / "processed" / "tokopedia_reviews_clean.csv"
FIG_DIR      = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Model dir    : {MODEL_DIR}")
print(f"Data path    : {DATA_PATH}")

if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Model not found at {MODEL_DIR}. Run 03_modeling.ipynb first to train and save the model."
    )
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Make sure project folder contains data/processed/tokopedia_reviews_clean.csv"
    )

DEVICE = torch.device("cpu")   # keep CPU for explainability (SHAP needs it)

# Load saved model
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model     = model.eval().to(DEVICE)

with open(MODEL_DIR / "label_map.json") as f:
    label_map = json.load(f)
ID2LABEL = {int(k): v for k, v in label_map["id2label"].items()}
CLASS_NAMES = [ID2LABEL[i] for i in range(2)]

print("Model loaded. Classes:", CLASS_NAMES)


In [ ]:
## 3. Prediction Helper & Sample Selection

MAX_LEN = 128

def predict_proba(texts):
    """Return probability array (N, 3) for a list of strings."""
    enc = tokenizer(
        list(texts), max_length=MAX_LEN, padding=True,
        truncation=True, return_tensors="pt"
    )
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

# Load data and pick sample reviews
df = pd.read_csv(DATA_PATH)
df["label_id"] = df["sentiment_label"].map({"Positif": 0, "Negatif": 1, "Netral": 1})

# Select 2 samples per class for explanation
samples = (
    df.groupby("sentiment_label")
      .apply(lambda g: g.sample(min(2, len(g)), random_state=42))
      .reset_index(drop=True)
)
print(f"Samples selected: {len(samples)}")
print(samples[["sentiment_label", "review_text_clean"]].head(6).to_string(index=False))


In [ ]:
## 4. LIME Explanations

explainer_lime = LimeTextExplainer(class_names=CLASS_NAMES)

lime_results = []        # store for comparison section
N_LIME_FEATURES = 10     # top features per explanation

print("=== LIME Explanations ===\n")
for _, row in samples.iterrows():
    text  = str(row["review_text_clean"])
    true_label = row["sentiment_label"]
    pred_probs = predict_proba([text])[0]
    pred_label = CLASS_NAMES[pred_probs.argmax()]

    exp = explainer_lime.explain_instance(
        text,
        predict_proba,
        num_features=N_LIME_FEATURES,
        num_samples=300,
        labels=[0, 1],
    )
    top_label_idx = pred_probs.argmax()
    feats = exp.as_list(label=top_label_idx)
    lime_results.append({"text": text, "true": true_label, "pred": pred_label,
                         "probs": pred_probs, "features": feats})

    print(f"True: {true_label:<8}  Pred: {pred_label}")
    print(f"Probs → {dict(zip(CLASS_NAMES, pred_probs.round(3)))}")
    print(f"Top tokens for '{pred_label}':")
    for token, weight in feats[:5]:
        print(f"  {token:25s} {weight:+.4f}")
    print()

print("LIME done.")


In [ ]:
## 5. LIME — Bar Chart Visualisation (one chart per sample)

for i, res in enumerate(lime_results):
    feats  = res["features"]
    tokens = [f[0] for f in feats]
    weights = [f[1] for f in feats]
    colors  = ["#2196F3" if w > 0 else "#F44336" for w in weights]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(tokens[::-1], weights[::-1], color=colors[::-1])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"LIME — Sample {i+1}  (True: {res['true']}  Pred: {res['pred']})", fontsize=11)
    ax.set_xlabel("LIME weight")
    pos_patch = mpatches.Patch(color="#2196F3", label="Supports prediction")
    neg_patch = mpatches.Patch(color="#F44336", label="Contradicts prediction")
    ax.legend(handles=[pos_patch, neg_patch], fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"04_lime_sample{i+1}.png", dpi=150, bbox_inches="tight")
    plt.show()

print("LIME charts saved.")

In [ ]:
## 6. SHAP Explanations (PartitionExplainer — NLP-friendly)

# Build SHAP explainer using a small background corpus
background_texts = df["review_text_clean"].sample(50, random_state=42).tolist()

# Wrapper: returns (N, 3) probabilities
def shap_predict(texts):
    return predict_proba([str(t) for t in texts])

masker   = shap.maskers.Text(tokenizer=" ")
explainer_shap = shap.Explainer(shap_predict, masker, output_names=CLASS_NAMES)

shap_texts = [str(r["review_text_clean"]) for _, r in samples.iterrows()]
shap_values = explainer_shap(shap_texts)

print(f"SHAP values computed for {len(shap_texts)} samples.")
print(f"Shape: {shap_values.values.shape}  (samples × tokens × classes)")

In [ ]:
## 7. SHAP — Waterfall Plots (per sample, predicted class)

for i, (_, row) in enumerate(samples.iterrows()):
    pred_probs = predict_proba([str(row["review_text_clean"])])[0]
    pred_class_idx = pred_probs.argmax()

    sv = shap_values[i, :, pred_class_idx]
    plt.figure()
    shap.waterfall_plot(sv, max_display=12, show=False)
    plt.title(f"SHAP Waterfall — Sample {i+1}  (True:{row['sentiment_label']}  Pred:{CLASS_NAMES[pred_class_idx]})")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"04_shap_waterfall_sample{i+1}.png", dpi=150, bbox_inches="tight")
    plt.show()

print("SHAP waterfall charts saved.")

In [ ]:
## 8. Global SHAP — Bar Summary (mean |SHAP| per class)

# Use a slightly larger set for global analysis
global_texts = (
    df.groupby("sentiment_label")
      .apply(lambda g: g["review_text_clean"].sample(min(10, len(g)), random_state=0))
      .reset_index(drop=True).tolist()
)
global_sv = explainer_shap([str(t) for t in global_texts])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for c_idx, c_name in enumerate(CLASS_NAMES):
    vals = np.abs(global_sv.values[:, :, c_idx])
    # aggregate over samples: mean per token position is not meaningful across different lengths
    # use cohorts_plot or beeswarm instead — keep it simple with bar
    mean_abs = vals.mean(axis=0)
    top_k = 12
    # global_sv.data gives list-of-list tokens; flatten for ordering is tricky; skip for bar
    axes[c_idx].set_title(f"Global mean |SHAP| — {c_name}")
    axes[c_idx].set_xlabel("|SHAP value|")

shap.plots.bar(global_sv[:, :, 0], max_display=12, show=False)
plt.suptitle("Global SHAP — Positif", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_shap_global_positif.png", dpi=150, bbox_inches="tight")
plt.show()

print("Global SHAP chart saved.")


In [ ]:
## 9. LIME vs SHAP Comparison Table

print("=== LIME vs SHAP: Top tokens comparison per sample ===\n")

for i, (res, (_, row)) in enumerate(zip(lime_results, samples.iterrows())):
    pred_probs = predict_proba([str(row["review_text_clean"])])[0]
    pred_class_idx = pred_probs.argmax()
    pred_class_name = CLASS_NAMES[pred_class_idx]

    # LIME top tokens
    lime_tokens = [f[0] for f in res["features"][:5]]

    # SHAP top tokens for same predicted class
    sv_i = shap_values[i, :, pred_class_idx]
    token_weights = list(zip(sv_i.data, sv_i.values))
    token_weights_sorted = sorted(token_weights, key=lambda x: abs(x[1]), reverse=True)
    shap_tokens = [t for t, w in token_weights_sorted[:5]]

    print(f"Sample {i+1}  (True: {res['true']}  Pred: {pred_class_name})")
    print(f"  LIME top-5 : {lime_tokens}")
    print(f"  SHAP top-5 : {shap_tokens}")
    overlap = set(lime_tokens) & set(shap_tokens)
    print(f"  Overlap    : {sorted(overlap) if overlap else '(none)'}\n")

## 10. Summary: LIME vs SHAP

| Aspect | LIME | SHAP |
|--------|------|------|
| **Theory** | Local linear surrogate | Shapley values (game theory) |
| **Speed** | Fast (few hundred perturbations) | Slower (exponential in theory, approx in practice) |
| **Consistency** | May vary across runs | Theoretically consistent |
| **Token-level** | Word/token level weights | Token-level Shapley values |
| **Global view** | Not directly | Yes (mean &#124;SHAP&#124; plot) |
| **Best for** | Quick local explanations | Rigorous local + global analysis |

**Key findings from this notebook:**
- Both methods highlight sentiment-bearing words (positive adjectives, complaint words)
- LIME is faster but less stable; SHAP provides richer token attribution
- Minority class (Negatif/Netral) explanations show the model focuses on negation words and complaint phrases